# Phase 1 Prep — Delta Retention

Extends log retention on all 12 Silver output tables to 60 days so Delta Time Travel
is available for the Workstream 6 validation if needed.

Attach to: APAC_Reporting_LH

Run once before any Phase 1 code changes.

## Cell 1 — Discover output table names

Run this first. Confirm the 6 country split table names and update OUTPUT_TABLES in Cell 2.

In [ ]:
tables = spark.sql("SHOW TABLES IN APAC_Reporting_LH").toPandas()

mask = (
    tables["tableName"].str.startswith("CRB")
    | tables["tableName"].str.startswith("APAC")
    | tables["tableName"].str.startswith("DIM")
)

print(tables[mask][["tableName"]].to_string(index=False))


## Cell 2 — Set retention to 60 days

Update the country split names below with the exact names from Cell 1 before running.

In [ ]:
OUTPUT_TABLES = [
    # APAC Sales Model outputs
    "APAC_Sales_Pipeline_FACT",
    "DIM_Account",
    "DIM_Segment",
    # CRB notebook outputs
    "APAC_CRB_ALL_RECORDS",
    "CRB_New_Business_Pipeline",
    "CRB_Renewal_Base",
    # Country splits — verify exact names from Cell 1
    "CRB_Singapore",
    "CRB_Hong_Kong",
    "CRB_India",
    "CRB_Taiwan",
    "CRB_Philippines",
    "CRB_China",
]

for t in OUTPUT_TABLES:
    spark.sql(f"""
        ALTER TABLE APAC_Reporting_LH.{t}
        SET TBLPROPERTIES (
            'delta.logRetentionDuration'         = 'interval 60 days',
            'delta.deletedFileRetentionDuration' = 'interval 60 days'
        )
    """)
    print(f"OK  {t}")
